In [1]:
"""
Logistics Database — Cleaning Script
=====================================

Based on the audit results: FK integrity is perfect, range checks are
clean, and date coverage has no gaps. So this cleaning pass is mostly
about (1) type-casting date columns that were loaded as text and
(2) documenting/validating the nulls that already showed up, rather
than fixing broken data.

HOW TO USE
----------
1. Edit FOLDER and OUTPUT_FOLDER below.
2. Run: python cleaning.py
3. Paste the printed summary back — flag anything under a
   "[REVIEW]" tag, since those are judgment calls, not auto-fixes.

WHAT THIS DOES
--------------
1. Loads all 14 tables
2. Parses every known date/datetime column to proper datetime dtype
3. Validates that FK nulls in trips/fuel_purchases line up with
   trip_status (e.g. Cancelled) rather than being unexplained gaps
4. Flags the single-row nulls in safety_incidents for manual review
5. Drops exact duplicate rows (defensive — audit found none)
6. Writes cleaned CSVs to OUTPUT_FOLDER, ready for the PostgreSQL load
"""

import pandas as pd
from pathlib import Path

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)

# ============================================================
# CONFIG — edit to match your paths
# ============================================================

FOLDER = Path(r"C:\Projects\3Year Truck Operations\data")          # raw CSVs
OUTPUT_FOLDER = Path(r"C:\Projects\3Year Truck Operations\data\clean")  # cleaned output

FILES = {
    "drivers": "drivers.csv",
    "trucks": "trucks.csv",
    "trailers": "trailers.csv",
    "customers": "customers.csv",
    "facilities": "facilities.csv",
    "routes": "routes.csv",
    "loads": "loads.csv",
    "trips": "trips.csv",
    "fuel_purchases": "fuel_purchases.csv",
    "maintenance_records": "maintenance_records.csv",
    "delivery_events": "delivery_events.csv",
    "safety_incidents": "safety_incidents.csv",
    "driver_monthly_metrics": "driver_monthly_metrics.csv",
    "truck_utilization_metrics": "truck_utilization_metrics.csv",
}

# Date/datetime columns per table, confirmed from the audit's [CHECK] flags
DATE_COLUMNS = {
    "drivers": ["hire_date", "termination_date", "date_of_birth"],
    "trucks": ["acquisition_date"],
    "trailers": ["acquisition_date"],
    "customers": ["contract_start_date"],
    "loads": ["load_date"],
    "trips": ["dispatch_date"],
    "fuel_purchases": ["purchase_date"],
    "maintenance_records": ["maintenance_date"],
    "delivery_events": ["scheduled_datetime", "actual_datetime"],
    "safety_incidents": ["incident_date"],
    "driver_monthly_metrics": ["month"],
    "truck_utilization_metrics": ["month"],
}

# Columns where null is EXPECTED and should be left as null, not filled
EXPECTED_NULL_COLUMNS = {
    "drivers": ["termination_date"],  # null = still employed
}


# ============================================================
# LOAD
# ============================================================

def load_tables():
    tables = {}
    for name, filename in FILES.items():
        path = FOLDER / filename
        if not path.exists():
            print(f"  [SKIP] {name}: not found at {path}")
            continue
        tables[name] = pd.read_csv(path, low_memory=False)
    return tables


# ============================================================
# DATE PARSING
# ============================================================

def parse_dates(tables):
    print("\n" + "=" * 70)
    print("PARSING DATE COLUMNS")
    print("=" * 70)
    for name, cols in DATE_COLUMNS.items():
        if name not in tables:
            continue
        df = tables[name]
        for col in cols:
            if col not in df.columns:
                print(f"[{name}] [REVIEW] expected date column '{col}' not found")
                continue
            before_nonnull = df[col].notna().sum()
            df[col] = pd.to_datetime(df[col], errors="coerce")
            after_nonnull = df[col].notna().sum()
            lost = before_nonnull - after_nonnull
            if lost:
                print(f"[{name}] '{col}' -> datetime. {lost} values failed to parse (became NaT).")
                print(f"  [REVIEW] {lost} unparseable '{col}' values in {name} — inspect before loading")
            else:
                print(f"[{name}] '{col}' -> datetime. All values parsed cleanly.")


# ============================================================
# NULL VALIDATION
# ============================================================

def validate_nulls(tables):
    print("\n" + "=" * 70)
    print("NULL VALIDATION")
    print("=" * 70)

    # drivers.termination_date — expected, no action
    if "drivers" in tables:
        df = tables["drivers"]
        if "termination_date" in df.columns:
            pct = df["termination_date"].isna().mean() * 100
            print(f"[drivers] termination_date null: {pct:.2f}% — expected (still employed), left as-is")

    # trips: driver_id/truck_id/trailer_id nulls should align with trip_status
    if "trips" in tables:
        df = tables["trips"]
        status_col = "trip_status" if "trip_status" in df.columns else None
        for fk in ["driver_id", "truck_id", "trailer_id"]:
            if fk not in df.columns:
                continue
            null_rows = df[df[fk].isna()]
            if len(null_rows) == 0:
                continue
            if status_col:
                statuses = null_rows[status_col].value_counts()
                non_cancelled = null_rows[~null_rows[status_col].astype(str).str.contains(
                    "cancel", case=False, na=False)]
                print(f"[trips] '{fk}' nulls ({len(null_rows)}) by status:\n{statuses.to_string()}")
                if len(non_cancelled):
                    print(f"  [REVIEW] {len(non_cancelled)} of these nulls are NOT on a cancelled-looking "
                          f"status — inspect these rows before assuming they're benign")
                else:
                    print(f"  [OK] all '{fk}' nulls fall on a cancelled-looking status — expected, left as-is")
            else:
                print(f"[trips] [REVIEW] '{fk}' has {len(null_rows)} nulls but no trip_status column "
                      f"to validate against")

    # fuel_purchases: same pattern, cross-check against trips.trip_status via trip_id
    if "fuel_purchases" in tables and "trips" in tables:
        fp = tables["fuel_purchases"]
        trips = tables["trips"]
        if "trip_id" in fp.columns and "trip_status" in trips.columns:
            merged = fp.merge(trips[["trip_id", "trip_status"]], on="trip_id", how="left")
            for fk in ["driver_id", "truck_id"]:
                if fk not in fp.columns:
                    continue
                null_rows = merged[merged[fk].isna()]
                if len(null_rows) == 0:
                    continue
                non_cancelled = null_rows[~null_rows["trip_status"].astype(str).str.contains(
                    "cancel", case=False, na=False)]
                print(f"[fuel_purchases] '{fk}' nulls ({len(null_rows)}): "
                      f"{len(non_cancelled)} not on a cancelled-looking trip")
                if len(non_cancelled):
                    print(f"  [REVIEW] inspect these — fuel was purchased but driver/truck ID missing "
                          f"on a non-cancelled trip")

    # safety_incidents: single-row nulls — flag for manual review
    if "safety_incidents" in tables:
        df = tables["safety_incidents"]
        for fk in ["driver_id", "truck_id"]:
            if fk in df.columns:
                null_rows = df[df[fk].isna()]
                if len(null_rows):
                    print(f"[safety_incidents] [REVIEW] '{fk}' null on {len(null_rows)} row(s) — "
                          f"manually inspect (small table, worth a direct look):")
                    print(null_rows.to_string())


# ============================================================
# DUPLICATES
# ============================================================

def drop_duplicates(tables):
    print("\n" + "=" * 70)
    print("DUPLICATE REMOVAL")
    print("=" * 70)
    for name, df in tables.items():
        before = len(df)
        df.drop_duplicates(inplace=True)
        after = len(df)
        removed = before - after
        if removed:
            print(f"[{name}] removed {removed:,} exact duplicate rows")
        else:
            print(f"[{name}] no duplicates found")


# ============================================================
# SAVE
# ============================================================

def save_clean(tables):
    print("\n" + "=" * 70)
    print("SAVING CLEANED FILES")
    print("=" * 70)
    OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
    for name, df in tables.items():
        out_path = OUTPUT_FOLDER / f"{name}_clean.csv"
        df.to_csv(out_path, index=False)
        print(f"[{name}] saved {len(df):,} rows -> {out_path}")


# ============================================================
# MAIN
# ============================================================

def main():
    print("Loading tables from:", FOLDER.resolve())
    tables = load_tables()
    print(f"Loaded {len(tables)}/{len(FILES)} tables successfully.")

    parse_dates(tables)
    validate_nulls(tables)
    drop_duplicates(tables)
    save_clean(tables)

    print("\n" + "=" * 70)
    print("DONE — review any [REVIEW] lines above before loading into PostgreSQL.")
    print("=" * 70)


if __name__ == "__main__":
    main()

Loading tables from: C:\Projects\3Year Truck Operations\data
Loaded 14/14 tables successfully.

PARSING DATE COLUMNS
[drivers] 'hire_date' -> datetime. All values parsed cleanly.
[drivers] 'termination_date' -> datetime. All values parsed cleanly.
[drivers] 'date_of_birth' -> datetime. All values parsed cleanly.
[trucks] 'acquisition_date' -> datetime. All values parsed cleanly.
[trailers] 'acquisition_date' -> datetime. All values parsed cleanly.
[customers] 'contract_start_date' -> datetime. All values parsed cleanly.
[loads] 'load_date' -> datetime. All values parsed cleanly.
[trips] 'dispatch_date' -> datetime. All values parsed cleanly.
[fuel_purchases] 'purchase_date' -> datetime. All values parsed cleanly.
[maintenance_records] 'maintenance_date' -> datetime. All values parsed cleanly.
[delivery_events] 'scheduled_datetime' -> datetime. All values parsed cleanly.
[delivery_events] 'actual_datetime' -> datetime. All values parsed cleanly.
[safety_incidents] 'incident_date' -> dat